# 10K Row Scaling Benchmark

- **Full Gibbs sweep** on 10K x 20 (all continuous)
- **Subsample init + batch insert**: init on 1K rows, insert remaining 9K
- **Insert throughput**: `packed_insert_rows` speed measurement

Min VRAM: 8GB, Recommended: T4 (16GB).

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}

try:
    import crosscat

    print(f"crosscat {crosscat.__version__} already installed")
except ImportError:
    if "COLAB_RELEASE_TAG" in os.environ:
        WORKDIR = "/content/jaxcross"
    elif Path("/kaggle/working").exists():
        WORKDIR = "/kaggle/working/jaxcross"
    else:
        WORKDIR = str(Path.home() / "jaxcross")
    if not Path(WORKDIR).exists():
        subprocess.run(
            ["git", "clone", "https://github.com/sambhal-labs/jaxcross.git", WORKDIR],
            check=True,
        )
    subprocess.run(["git", "fetch", "origin"], cwd=WORKDIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=WORKDIR, check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", WORKDIR, "--no-deps", "-q"],
        check=True,
    )
    os.chdir(WORKDIR)

In [ ]:
import json
import time

import jax
import jax.numpy as jnp

from benchmarks.utils import detect_platform, make_benchmark_data
from crosscat import (
    initialize,
    pack_state,
    packed_gibbs_sweep,
    packed_insert_rows,
    suggest_max_clusters,
)

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")
print(f"suggest_max_clusters(10000) = {suggest_max_clusters(10000)}")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Full Gibbs Sweep (10K rows x 20 cols)

Direct `packed_gibbs_sweep` on the full dataset.

In [ ]:
print("--- Full Gibbs Sweep: 10,000 rows x 20 cols ---")
k1, k2, k3 = jax.random.split(jax.random.key(42), 3)
data, col_types = make_benchmark_data(k1, 10_000, 20)
max_k = suggest_max_clusters(10_000)
print(f"  max_clusters: {max_k}, data: {data.nbytes / (1024 * 1024):.1f} MB")

t0 = time.perf_counter()
state = initialize(k2, data, col_types).state
packed = pack_state(state, max_clusters=max_k)
print(f"  init + pack: {time.perf_counter() - t0:.2f}s")

# JIT compile
t0 = time.perf_counter()
packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=1)
packed.column_assignments.block_until_ready()
print(f"  JIT compile (1st sweep): {time.perf_counter() - t0:.2f}s")

# Timed sweeps
n_sweeps = 5
k4 = jax.random.fold_in(k3, 1)
t0 = time.perf_counter()
packed = packed_gibbs_sweep(k4, packed, data, n_sweeps=n_sweeps)
packed.column_assignments.block_until_ready()
sweep_time = time.perf_counter() - t0
per_sweep = sweep_time / n_sweeps
print(f"  {n_sweeps} sweeps: {sweep_time:.2f}s ({per_sweep:.2f}s/sweep)")

## 3. Subsample Init + Batch Insert

Initialize on 1K rows, then insert remaining 9K in batches.

In [ ]:
print("--- Subsample Init: 1000 init, 10000 total ---")
k1, k2, k3, k4 = jax.random.split(jax.random.key(43), 4)
data, col_types = make_benchmark_data(k1, 10_000, 20)
max_k = suggest_max_clusters(10_000)

t0 = time.perf_counter()
result = initialize(k2, data, col_types, subsample_rows=1000)
sub_idx = result.subsample_idx
sub_data = data[sub_idx]
packed = pack_state(result.state, max_clusters=max_k)
print(f"  subsample init + pack: {time.perf_counter() - t0:.2f}s ({packed.n_rows} rows)")

t0 = time.perf_counter()
packed = packed_gibbs_sweep(k3, packed, sub_data, n_sweeps=5)
packed.column_assignments.block_until_ready()
print(f"  5 pre-sweeps on subsample: {time.perf_counter() - t0:.2f}s")

# Insert remaining
remaining_mask = jnp.ones(10_000, dtype=bool).at[sub_idx].set(False)
remaining_idx = jnp.where(remaining_mask, size=10_000 - 1000)[0]
remaining_data = data[remaining_idx]
batch_size = 1000
n_batches = (remaining_data.shape[0] + batch_size - 1) // batch_size

t0 = time.perf_counter()
current_data = sub_data
for b in range(n_batches):
    batch = remaining_data[b * batch_size : (b + 1) * batch_size]
    kb = jax.random.fold_in(k4, b)
    packed, current_data = packed_insert_rows(kb, packed, current_data, batch)
insert_time = time.perf_counter() - t0
print(f"  inserted {remaining_data.shape[0]} rows in {n_batches} batches: {insert_time:.2f}s")
print(f"  final n_rows: {packed.n_rows}")

k5 = jax.random.fold_in(k4, 999)
t0 = time.perf_counter()
packed = packed_gibbs_sweep(k5, packed, current_data, n_sweeps=3)
packed.column_assignments.block_until_ready()
print(f"  3 post-sweeps on full data: {time.perf_counter() - t0:.2f}s")

## 4. Insert Throughput

Measure `packed_insert_rows` speed: insert 5000 rows into a 1000-row base.

In [ ]:
print("--- Insert Throughput: 5000 rows into 1000-row base ---")
k1, k2, k3, k4 = jax.random.split(jax.random.key(44), 4)
base_data, col_types = make_benchmark_data(k1, 1000, 20)
new_rows, _ = make_benchmark_data(k2, 5000, 20)
max_k = suggest_max_clusters(6000)

state = initialize(k3, base_data, col_types).state
packed = pack_state(state, max_clusters=max_k)
packed = packed_gibbs_sweep(jax.random.fold_in(k3, 1), packed, base_data, n_sweeps=5)
packed.column_assignments.block_until_ready()

batch_size = 500
n_batches = (5000 + batch_size - 1) // batch_size
current_data = base_data
t0 = time.perf_counter()
for b in range(n_batches):
    batch = new_rows[b * batch_size : (b + 1) * batch_size]
    kb = jax.random.fold_in(k4, b)
    packed, current_data = packed_insert_rows(kb, packed, current_data, batch)
total = time.perf_counter() - t0
rows_per_sec = 5000 / total
print(f"  5000 rows in {total:.2f}s ({rows_per_sec:.0f} rows/s)")
print(f"  final n_rows: {packed.n_rows}")

## 5. Save Results

In [ ]:
import shutil

results_dir = Path("benchmarks/results/scaling")
results_dir.mkdir(parents=True, exist_ok=True)

results_10k = {
    "backend": platform["backend"],
    "device": platform["gpu_names"],
    "per_sweep_10k": per_sweep,
    "insert_throughput": rows_per_sec,
}
with open(results_dir / "scaling_10k_results.json", "w") as f:
    json.dump(results_10k, f, indent=2)

print(f"Results saved to {results_dir / 'scaling_10k_results.json'}")
archive = Path("benchmarks/results/scaling_10k_results")
shutil.make_archive(str(archive), "gztar", ".", str(results_dir))
print(f"Archived to {archive}.tar.gz")